In [69]:
import pandas as pd
import numpy as np
import sqlite3
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer,MissingIndicator, IterativeImputer


# !pip install ydata-profiling
# from ydata_profiling import ProfileReport

#### Part A - Conceptual Foundation

#### Part B - Data Acquisition


In [45]:
# Import datasets
# csv
df_csv = pd.read_csv('transactions_main.csv')
display(df_csv.head())

,customer_id,loan_amount,loan_purpose,transaction_count,spending_ratio,default_flag
0,C00001,9160.701553,Home,143,0.772072,0
1,C00002,40556.429836,Business,208,0.588044,0
2,C00003,54039.038178,Home,247,0.662325,0
3,C00004,18820.998791,Home,453,0.395924,0
4,C00005,14495.124874,Education,121,0.167524,0


In [46]:
# json
df_json = pd.read_json('customer_metadata.json')
display(df_json.head())

,customer_id,age,gender,region,education_level,employment_type,join_date
0,C00001,NaN,Male,East,Secondary,Salaried,2021-01-31
1,C00002,30.0,NaN,West,Primary,Self-Employed,2021-12-30
2,C00003,60.0,Male,East,Graduate,Salaried,2020-05-10
3,C00004,52.0,Female,South,Post-Graduate,Salaried,2021-07-18
4,C00005,70.0,Male,East,Post-Graduate,Self-Employed,2021-02-04


In [47]:
# sql
conn = sqlite3.connect('financial_history.db')

table = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(f'Table Name: {table['name'][0]}')

df_sql = pd.read_sql('SELECT * FROM repayment_records', conn)
display(df_sql.head())

Table Name: repayment_records


,customer_id,annual_income,credit_score,repayment_history
0,C00001,66220.926969,686.739425,0
1,C00002,45743.670179,164.529051,2
2,C00003,88735.232342,697.387262,2
3,C00004,90609.732170,587.777427,4
4,C00005,59077.776173,428.924467,3


In [48]:
# Merge

df = df_csv.merge(df_json, on='customer_id').merge(df_sql, on='customer_id')
df

,customer_id,loan_amount,loan_purpose,transaction_count,spending_ratio,default_flag,age,gender,region,education_level,employment_type,join_date,annual_income,credit_score,repayment_history
0,C00001,9160.701553,Home,143,0.772072,0,NaN,Male,East,Secondary,Salaried,2021-01-31,66220.926969,686.739425,0
1,C00002,40556.429836,Business,208,0.588044,0,30.0,NaN,West,Primary,Self-Employed,2021-12-30,45743.670179,164.529051,2
2,C00003,54039.038178,Home,247,0.662325,0,60.0,Male,East,Graduate,Salaried,2020-05-10,88735.232342,697.387262,2
3,C00004,18820.998791,Home,453,0.395924,0,52.0,Female,South,Post-Graduate,Salaried,2021-07-18,90609.732170,587.777427,4
4,C00005,14495.124874,Education,121,0.167524,0,70.0,Male,East,Post-Graduate,Self-Employed,2021-02-04,59077.776173,428.924467,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2495,C02496,6731.263574,Business,444,0.330863,0,74.0,Female,West,Primary,Unemployed,2019-08-15,79227.316574,682.413804,2
2496,C02497,41407.355017,Car,80,0.673862,1,67.0,NaN,North,Primary,Salaried,2021-08-26,63138.631982,979.606514,0
2497,C02498,44510.359447,Business,391,0.236262,0,51.0,Male,North,Secondary,Salaried,2021-01-14,55461.026960,606.108035,2
2498,C02499,156731.479266,Business,68,0.483820,0,61.0,Male,East,Secondary,NaN,2018-04-02,60389.475345,585.851512,2


#### Part C - Data Understanding & Cleaning

In [50]:
# Fix the arrangement of columns

df = df[['customer_id','age','gender','region','education_level','employment_type','annual_income','loan_amount','loan_purpose','credit_score','repayment_history','transaction_count','spending_ratio','join_date','default_flag']]
display(df.head())

,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,repayment_history,transaction_count,spending_ratio,join_date,default_flag
0,C00001,NaN,Male,East,Secondary,Salaried,66220.926969,9160.701553,Home,686.739425,0,143,0.772072,2021-01-31,0
1,C00002,30.0,NaN,West,Primary,Self-Employed,45743.670179,40556.429836,Business,164.529051,2,208,0.588044,2021-12-30,0
2,C00003,60.0,Male,East,Graduate,Salaried,88735.232342,54039.038178,Home,697.387262,2,247,0.662325,2020-05-10,0
3,C00004,52.0,Female,South,Post-Graduate,Salaried,90609.732170,18820.998791,Home,587.777427,4,453,0.395924,2021-07-18,0
4,C00005,70.0,Male,East,Post-Graduate,Self-Employed,59077.776173,14495.124874,Education,428.924467,3,121,0.167524,2021-02-04,0


In [57]:
# info
print('Info: ')
print({df.info()})

Info: 
<class 'pandas.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        2500 non-null   str    
 1   age                2200 non-null   float64
 2   gender             2200 non-null   str    
 3   region             2500 non-null   str    
 4   education_level    2500 non-null   str    
 5   employment_type    2200 non-null   str    
 6   annual_income      2215 non-null   float64
 7   loan_amount        2500 non-null   float64
 8   loan_purpose       2500 non-null   str    
 9   credit_score       2212 non-null   float64
 10  repayment_history  2500 non-null   int64  
 11  transaction_count  2500 non-null   int64  
 12  spending_ratio     2500 non-null   float64
 13  join_date          2500 non-null   str    
 14  default_flag       2500 non-null   int64  
dtypes: float64(5), int64(3), str(7)
memory usage: 293.1 KB
{None}


In [66]:
print('Describe: ')
print(df.describe().round(2).to_string())

Describe: 
           age  annual_income  loan_amount  credit_score  repayment_history  transaction_count  spending_ratio  default_flag
count  2200.00        2215.00      2500.00       2212.00            2500.00            2500.00         2500.00        2500.0
mean     47.42       74053.32     30536.45        650.07               1.55             250.20            0.45           0.2
std      15.46       71715.35     28750.62        121.88               1.22             143.12            0.20           0.4
min      21.00      -18535.36       884.22        110.30               0.00              10.00            0.10           0.0
25%      34.00       43258.27     12815.47        593.31               1.00             123.75            0.27           0.0
50%      47.00       61397.80     21754.09        652.63               1.00             248.00            0.45           0.0
75%      61.00       79361.85     38471.67        712.12               2.00             377.00            0.62    

In [67]:
print('Null Values Count:')
print(df.isnull().sum())

Null Values Count:
customer_id            0
age                  300
gender               300
region                 0
education_level        0
employment_type      300
annual_income        285
loan_amount            0
loan_purpose           0
credit_score         288
repayment_history      0
transaction_count      0
spending_ratio         0
join_date              0
default_flag           0
dtype: int64


In [ ]:
# ProfileReport

# profile = ProfileReport(df, title='Customer Credit Risk Dataset EDA Report', explorative=True)
#
# profile.to_file('titanic_eda.html')

In [70]:
df_missing = df[['age','gender','employment_type','annual_income','credit_score']]

#### Part D - Outlier Handling

#### Part E - Feature Engineering

#### Part F - Feature Scaling

#### Part G - Feature Construction & Transformation

#### Part H - Final Deliverable